# Authors model

## 1 step prediction

In [1]:
import torch
from torch.utils.data import DataLoader
from einops import rearrange
from tqdm import tqdm
from the_well.benchmark.models import FNO
from the_well.data import WellDataset
from the_well.data.normalization import ZScoreNormalization
from the_well.benchmark.metrics import VRMSE

base_path='./data'
device=torch.device('mps')
model = FNO.from_pretrained('polymathic-ai/FNO-turbulent_radiative_layer_2D').to(device).eval()

dset = WellDataset(
    well_base_path=f'{base_path}/datasets',
    well_dataset_name='turbulent_radiative_layer_2D',
    well_split_name='test',
    n_steps_input=4,
    n_steps_output=1,
    use_normalization=True,
    normalization_type=ZScoreNormalization,
)
loader = DataLoader(dset, batch_size=8, shuffle=False)
F = dset.metadata.n_fields
norm = dset.norm
mean_full = norm.flattened_means['variable'].to(device)       # (F,)
std_full  = norm.flattened_stds['variable'].to(device)
mean_d    = norm.flattened_means_delta['variable'].to(device)
std_d     = norm.flattened_stds_delta['variable'].to(device)

sum_err2 = torch.zeros(F, device=device)
sum_y = torch.zeros(F, device=device)
sum_y2 = torch.zeros(F, device=device)
count = 0

sum_abs_err = torch.zeros(F, device=device)

rel_l2_num = torch.zeros(F, device=device)
rel_l2_den = torch.zeros(F, device=device)

with torch.no_grad():
    for batch in tqdm(loader, desc='stream accum'):
        x = batch['input_fields'].to(device)
        y = batch['output_fields'].to(device)
        # model expects normalized inputs already because dset uses normalization
        x_in = rearrange(x, 'B Ti Lx Ly F -> B (Ti F) Lx Ly')
        pred = model(x_in)
        pred = rearrange(pred, 'B (To F) Lx Ly -> B To Lx Ly F', To=1, F=F)

        last_input_full_norm = x[:, -1]      # (B,Lx,Ly,F) normalized full
        pred_delta_norm = pred[:, 0]         # (B,Lx,Ly,F) delta normalized

        # delta -> full normalized inversion
        last_raw = last_input_full_norm * std_full + mean_full
        delta_raw = pred_delta_norm * std_d + mean_d
        full_raw = last_raw + delta_raw
        pred_full_norm = (full_raw - mean_full) / std_full
        pred_full_norm = pred_full_norm.unsqueeze(1)  # (B,To=1,Lx,Ly,F)

        err = pred_full_norm - y
        err2 = err ** 2

        sum_err2 += err2.sum(dim=(0,1,2,3))
        sum_y += y.sum(dim=(0,1,2,3))
        sum_y2 += (y*y).sum(dim=(0,1,2,3))
        
        sum_abs_err += err.abs().sum(dim=(0, 1, 2, 3))
        rel_l2_num += err2.sum(dim=(0, 1, 2, 3))
        rel_l2_den += (y ** 2).sum(dim=(0, 1, 2, 3))

        count += y.shape[0] * y.shape[1] * y.shape[2] * y.shape[3]

mse_f = sum_err2 / count
Ey_f = sum_y / count
Ey2_f = sum_y2 / count
Var_f = Ey2_f - Ey_f*Ey_f

vrmse_per_field = torch.sqrt(mse_f / (Var_f + 1e-7))
vrmse_mean_fields = vrmse_per_field.mean()

#additional metrics
# RMSE
rmse_f = torch.sqrt(mse_f)
# MAE
mae_f = sum_abs_err / count
# R^2
r2_f = 1.0 - (mse_f / (Var_f + 1e-7))
# Relative L2 error
rel_l2_f = torch.sqrt(rel_l2_num / (rel_l2_den + 1e-7))

rmse_mean = rmse_f.mean()
r2_mean = r2_f.mean()
rel_l2_mean = rel_l2_f.mean()

# Variant: compute MSE and Var after averaging over fields first
mse_mean = mse_f.mean()
Var_mean = Var_f.mean()
vrmse_fields_averaged_then_sqrt = torch.sqrt(mse_mean / (Var_mean + 1e-7))

# Variant: flatten fields too (treat each field equally by summing and dividing by F)
vrmse_flattened = torch.sqrt((mse_f.sum()) / (Var_f.sum() + 1e-7))

print('per-field VRMSE:', vrmse_per_field.tolist())
print('mean(per-field VRMSE):', vrmse_mean_fields.item())
print('sqrt(mean MSE / mean Var):', vrmse_fields_averaged_then_sqrt.item())
print('sqrt(sum MSE / sum Var):', vrmse_flattened.item())

print("\n=== Additional metrics ===")

print("RMSE per field:", rmse_f.tolist())
print("mean RMSE:", rmse_mean.item())

print("R2 per field:", r2_f.tolist())
print("mean R2:", r2_mean.item())

print("Relative L2 per field:", rel_l2_f.tolist())
print("mean Relative L2:", rel_l2_mean.item())

print("\n=== Summary ===")
print("VRMSE (mean):", vrmse_mean_fields.item())
print("Relative L2 (mean):", rel_l2_mean.item())
print("R2 (mean):", r2_mean.item())

/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
stream accum:   0%|          | 0/110 [00:00<?, ?it/s]/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/neuralop/layers/spectral_convolution.py:434: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [8, 128, 128, 193]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero el

per-field VRMSE: [0.3346270024776459, 0.6924504637718201, 0.46432241797447205, 0.5568460822105408]
mean(per-field VRMSE): 0.5120614767074585
sqrt(mean MSE / mean Var): 0.5320705771446228
sqrt(sum MSE / sum Var): 0.5320706367492676

=== Additional metrics ===
RMSE per field: [0.3329218029975891, 0.7133544087409973, 0.46642011404037476, 0.5780757665634155]
mean RMSE: 0.5226930379867554
R2 per field: [0.8880247473716736, 0.5205123424530029, 0.7844046950340271, 0.6899224519729614]
mean R2: 0.7207160592079163
Relative L2 per field: [0.334626704454422, 0.690365195274353, 0.46431684494018555, 0.5562203526496887]
mean Relative L2: 0.5113822817802429

=== Summary ===
VRMSE (mean): 0.5120614767074585
Relative L2 (mean): 0.5113822817802429
R2 (mean): 0.7207160592079163


## rollout prediction

In [14]:
import torch
from torch.utils.data import DataLoader
from einops import rearrange
from tqdm import tqdm
from the_well.benchmark.models import FNO
from the_well.data import WellDataset
from the_well.data.normalization import ZScoreNormalization

base_path = './data'
device = torch.device('mps')

model = FNO.from_pretrained(
    'polymathic-ai/FNO-turbulent_radiative_layer_2D'
).to(device).eval()

N_STEPS_INPUT = 4
N_STEPS_ROLLOUT = 4

dset = WellDataset(
    well_base_path=f'{base_path}/datasets',
    well_dataset_name='turbulent_radiative_layer_2D',
    well_split_name='test',
    n_steps_input=N_STEPS_INPUT,
    n_steps_output=N_STEPS_ROLLOUT,
    use_normalization=True,
    normalization_type=ZScoreNormalization,
)

loader = DataLoader(dset, batch_size=8, shuffle=False)

F = dset.metadata.n_fields
norm = dset.norm

mean_full = norm.flattened_means['variable'].to(device)         # (F,)
std_full  = norm.flattened_stds['variable'].to(device)          # (F,)
mean_d    = norm.flattened_means_delta['variable'].to(device)   # (F,)
std_d     = norm.flattened_stds_delta['variable'].to(device)    # (F,)

# -------------------------
# global accumulators over all rollout steps
# -------------------------
sum_err2 = torch.zeros(F, device=device)
sum_y = torch.zeros(F, device=device)
sum_y2 = torch.zeros(F, device=device)
sum_abs_err = torch.zeros(F, device=device)
rel_l2_num = torch.zeros(F, device=device)
rel_l2_den = torch.zeros(F, device=device)
count = 0

# -------------------------
# per-horizon accumulators: step 1, 2, ..., 8
# -------------------------
sum_err2_h = torch.zeros(N_STEPS_ROLLOUT, F, device=device)
sum_y_h = torch.zeros(N_STEPS_ROLLOUT, F, device=device)
sum_y2_h = torch.zeros(N_STEPS_ROLLOUT, F, device=device)
count_h = torch.zeros(N_STEPS_ROLLOUT, device=device)

with torch.no_grad():
    for batch in tqdm(loader, desc='autoregressive rollout accum'):
        x = batch['input_fields'].to(device)    # (B, 4, Lx, Ly, F)
        y = batch['output_fields'].to(device)   # (B, 8, Lx, Ly, F)

        # rolling window of normalized full frames
        current_window = x.clone()              # (B, 4, Lx, Ly, F)

        preds = []

        for t in range(N_STEPS_ROLLOUT):
            # model input: normalized full frames
            x_in = rearrange(current_window, 'B Ti Lx Ly F -> B (Ti F) Lx Ly')
            pred = model(x_in)  # predicts normalized delta
            pred = rearrange(pred, 'B (To F) Lx Ly -> B To Lx Ly F', To=1, F=F)

            last_input_full_norm = current_window[:, -1]   # (B, Lx, Ly, F)
            pred_delta_norm = pred[:, 0]                   # (B, Lx, Ly, F)

            # delta norm -> delta raw -> next full raw -> next full norm
            last_raw = last_input_full_norm * std_full + mean_full
            delta_raw = pred_delta_norm * std_d + mean_d
            next_full_raw = last_raw + delta_raw
            next_full_norm = (next_full_raw - mean_full) / std_full  # (B, Lx, Ly, F)

            preds.append(next_full_norm)

            # slide the window: drop oldest, append prediction
            current_window = torch.cat(
                [current_window[:, 1:], next_full_norm.unsqueeze(1)],
                dim=1
            )

        pred_full_norm = torch.stack(preds, dim=1)  # (B, 8, Lx, Ly, F)

        err = pred_full_norm - y
        err2 = err ** 2

        # -------------------------
        # global accumulators
        # -------------------------
        sum_err2 += err2.sum(dim=(0, 1, 2, 3))
        sum_y += y.sum(dim=(0, 1, 2, 3))
        sum_y2 += (y * y).sum(dim=(0, 1, 2, 3))

        sum_abs_err += err.abs().sum(dim=(0, 1, 2, 3))
        rel_l2_num += err2.sum(dim=(0, 1, 2, 3))
        rel_l2_den += (y ** 2).sum(dim=(0, 1, 2, 3))

        count += y.shape[0] * y.shape[1] * y.shape[2] * y.shape[3]

        # -------------------------
        # per-horizon accumulators
        # -------------------------
        for t in range(N_STEPS_ROLLOUT):
            err_t = err[:, t:t+1]   # (B,1,Lx,Ly,F)
            y_t = y[:, t:t+1]

            sum_err2_h[t] += (err_t ** 2).sum(dim=(0, 1, 2, 3))
            sum_y_h[t] += y_t.sum(dim=(0, 1, 2, 3))
            sum_y2_h[t] += (y_t ** 2).sum(dim=(0, 1, 2, 3))
            count_h[t] += y_t.shape[0] * y_t.shape[1] * y_t.shape[2] * y_t.shape[3]

# -------------------------
# global metrics over all 8 rollout steps
# -------------------------
mse_f = sum_err2 / count
Ey_f = sum_y / count
Ey2_f = sum_y2 / count
Var_f = Ey2_f - Ey_f * Ey_f

vrmse_per_field = torch.sqrt(mse_f / (Var_f + 1e-7))
vrmse_mean_fields = vrmse_per_field.mean()

rmse_f = torch.sqrt(mse_f)
mae_f = sum_abs_err / count
r2_f = 1.0 - (mse_f / (Var_f + 1e-7))
rel_l2_f = torch.sqrt(rel_l2_num / (rel_l2_den + 1e-7))

rmse_mean = rmse_f.mean()
r2_mean = r2_f.mean()
rel_l2_mean = rel_l2_f.mean()

mse_mean = mse_f.mean()
Var_mean = Var_f.mean()
vrmse_fields_averaged_then_sqrt = torch.sqrt(mse_mean / (Var_mean + 1e-7))
vrmse_flattened = torch.sqrt(mse_f.sum() / (Var_f.sum() + 1e-7))

# -------------------------
# per-horizon metrics
# -------------------------
mse_h = sum_err2_h / count_h[:, None]
Ey_h = sum_y_h / count_h[:, None]
Ey2_h = sum_y2_h / count_h[:, None]
Var_h = Ey2_h - Ey_h * Ey_h

vrmse_per_field_h = torch.sqrt(mse_h / (Var_h + 1e-7))   # (8, F)
vrmse_mean_h = vrmse_per_field_h.mean(dim=1)             # (8,)

# -------------------------
# printing
# -------------------------
print('=== AUTOREGRESSIVE ROLLOUT (4 STEPS) ===')
print('per-field VRMSE:', vrmse_per_field.tolist())
print('mean(per-field VRMSE):', vrmse_mean_fields.item())
print('sqrt(mean MSE / mean Var):', vrmse_fields_averaged_then_sqrt.item())
print('sqrt(sum MSE / sum Var):', vrmse_flattened.item())

print("\n=== Additional metrics ===")
print("RMSE per field:", rmse_f.tolist())
print("mean RMSE:", rmse_mean.item())

print("MAE per field:", mae_f.tolist())
print("mean MAE:", mae_f.mean().item())

print("R2 per field:", r2_f.tolist())
print("mean R2:", r2_mean.item())

print("Relative L2 per field:", rel_l2_f.tolist())
print("mean Relative L2:", rel_l2_mean.item())

print("\n=== Per-horizon VRMSE ===")
for t in range(N_STEPS_ROLLOUT):
    print(f"step {t+1}: mean VRMSE = {vrmse_mean_h[t].item():.6f}, per-field = {vrmse_per_field_h[t].tolist()}")

print("\n=== Summary ===")
print("VRMSE (mean):", vrmse_mean_fields.item())
print("Relative L2 (mean):", rel_l2_mean.item())
print("R2 (mean):", r2_mean.item())

autoregressive rollout accum: 100%|██████████| 106/106 [00:36<00:00,  2.88it/s]

=== AUTOREGRESSIVE ROLLOUT (4 STEPS) ===
per-field VRMSE: [1.9597140550613403, 1.4642834663391113, 0.9464032053947449, 2.3836936950683594]
mean(per-field VRMSE): 1.688523530960083
sqrt(mean MSE / mean Var): 1.780089259147644
sqrt(sum MSE / sum Var): 1.7800893783569336

=== Additional metrics ===
RMSE per field: [1.9488236904144287, 1.5016504526138306, 0.9512321949005127, 2.475010871887207]
mean RMSE: 1.7191792726516724
MAE per field: [0.40085628628730774, 0.7884702086448669, 0.4858262836933136, 0.711233377456665]
mean MAE: 0.5965965390205383
R2 per field: [-2.8404791355133057, -1.1441259384155273, 0.1043209433555603, -4.681995391845703]
mean R2: -2.1405699253082275
Relative L2 per field: [1.9597125053405762, 1.4591447114944458, 0.9463819861412048, 2.3804686069488525]
mean Relative L2: 1.6864269971847534

=== Per-horizon VRMSE ===
step 1: mean VRMSE = 0.509786, per-field = [0.3321203589439392, 0.6941355466842651, 0.458349347114563, 0.5545388460159302]
step 2: mean VRMSE = 0.723757, per-

In [ ]:
# from autoregressive_pretrained_fno import predict_autoregressive_fno, save_rollout_plots

# out = predict_autoregressive_fno(
#     base_path="./data",
#     hdf5_path="./data/datasets/turbulent_radiative_layer_2D/data/test/turbulent_radiative_layer_tcool_0.32.hdf5",  # or None if one test .hdf5 exists
#     traj_id=0,
#     t_start=50,
#     t_roll=10,
#     device="mps",
# )
# # out["pred_raw"]  → (t_roll, H, W, F) in raw units
# # out["traj"]      → full trajectory
# save_rollout_plots(out, field="density", out_dir="./autoregressive_plots_nb")

/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/neuralop/layers/spectral_convolution.py:434: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [1, 128, 128, 193]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by r

'./autoregressive_plots_nb/traj000_density_pretrained_delta'

# My delta models

## 1 step prediction

In [21]:
import torch
from torch.utils.data import DataLoader
from einops import rearrange
from tqdm import tqdm
from neuralop.models import FNO as NeuralFNO
from the_well.data import WellDataset
from the_well.data.normalization import ZScoreNormalization


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def load_my_model(checkpoint_path, n_fields, device):
    ckpt = torch.load(checkpoint_path, map_location="cpu")

    if isinstance(ckpt, torch.nn.Module):
        model = ckpt
    else:
        state_dict = None
        if isinstance(ckpt, dict):
            for key in ("model_state_dict", "state_dict", "model", "net"):
                if key in ckpt and isinstance(ckpt[key], dict):
                    state_dict = ckpt[key]
                    break
            if state_dict is None and all(isinstance(v, torch.Tensor) for v in ckpt.values()):
                state_dict = ckpt

        if state_dict is None:
            raise ValueError(
                "Unsupported checkpoint format. Expected nn.Module or state_dict-like dict."
            )

        if any(k.startswith("module.") for k in state_dict.keys()):
            state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

        model = NeuralFNO(
            n_modes=(32, 32),
            in_channels=4 * n_fields,
            out_channels=1 * n_fields,
            hidden_channels=128,
            n_layers=4,
        )
        model.load_state_dict(state_dict, strict=True)

    return model.to(device).eval()


base_path = "./data"
checkpoint_path = "/Users/emilfahretdinov/msc_hse/models_trained/fno_delta_lr1e-3_20ep_nmodes32_nlayers4/best_by_valid_rollout_vrmse_delta.pt"
device = pick_device()
print(f"Using device: {device}")

dset = WellDataset(
    well_base_path=f"{base_path}/datasets",
    well_dataset_name="turbulent_radiative_layer_2D",
    well_split_name="test",
    n_steps_input=4,
    n_steps_output=1,
    use_normalization=True,
    normalization_type=ZScoreNormalization,
)
loader = DataLoader(dset, batch_size=8, shuffle=False)
F = dset.metadata.n_fields
model = load_my_model(checkpoint_path, n_fields=F, device=device)

norm = dset.norm
mean_full = norm.flattened_means['variable'].to(device)       # (F,)
std_full  = norm.flattened_stds['variable'].to(device)
mean_d    = norm.flattened_means_delta['variable'].to(device)
std_d     = norm.flattened_stds_delta['variable'].to(device)

sum_err2 = torch.zeros(F, device=device)
sum_y = torch.zeros(F, device=device)
sum_y2 = torch.zeros(F, device=device)
count = 0

sum_abs_err = torch.zeros(F, device=device)

rel_l2_num = torch.zeros(F, device=device)
rel_l2_den = torch.zeros(F, device=device)

with torch.no_grad():
    for batch in tqdm(loader, desc='stream accum'):
        x = batch['input_fields'].to(device)
        y = batch['output_fields'].to(device)
        # model expects normalized inputs already because dset uses normalization
        x_in = rearrange(x, 'B Ti Lx Ly F -> B (Ti F) Lx Ly')
        pred = model(x_in)
        pred = rearrange(pred, 'B (To F) Lx Ly -> B To Lx Ly F', To=1, F=F)

        last_input_full_norm = x[:, -1]      # (B,Lx,Ly,F) normalized full
        pred_delta_norm = pred[:, 0]         # (B,Lx,Ly,F) delta normalized

        # delta -> full normalized inversion
        last_raw = last_input_full_norm * std_full + mean_full
        delta_raw = pred_delta_norm * std_d + mean_d
        full_raw = last_raw + delta_raw
        pred_full_norm = (full_raw - mean_full) / std_full
        pred_full_norm = pred_full_norm.unsqueeze(1)  # (B,To=1,Lx,Ly,F)

        err = pred_full_norm - y
        err2 = err ** 2  # (B,1,Lx,Ly,F)

        sum_err2 += err2.sum(dim=(0,1,2,3))
        sum_y += y.sum(dim=(0,1,2,3))
        sum_y2 += (y*y).sum(dim=(0,1,2,3))

        sum_abs_err += err.abs().sum(dim=(0, 1, 2, 3))
        rel_l2_num += err2.sum(dim=(0, 1, 2, 3))
        rel_l2_den += (y ** 2).sum(dim=(0, 1, 2, 3))
        
        count += y.shape[0] * y.shape[1] * y.shape[2] * y.shape[3]

mse_f = sum_err2 / count
Ey_f = sum_y / count
Ey2_f = sum_y2 / count
Var_f = Ey2_f - Ey_f*Ey_f

vrmse_per_field = torch.sqrt(mse_f / (Var_f + 1e-7))
vrmse_mean_fields = vrmse_per_field.mean()

#additional metrics
# RMSE
rmse_f = torch.sqrt(mse_f)
# MAE
mae_f = sum_abs_err / count
# R^2
r2_f = 1.0 - (mse_f / (Var_f + 1e-7))
# Relative L2 error
rel_l2_f = torch.sqrt(rel_l2_num / (rel_l2_den + 1e-7))

rmse_mean = rmse_f.mean()
r2_mean = r2_f.mean()
rel_l2_mean = rel_l2_f.mean()

# Variant: compute MSE and Var after averaging over fields first
mse_mean = mse_f.mean()
Var_mean = Var_f.mean()
vrmse_fields_averaged_then_sqrt = torch.sqrt(mse_mean / (Var_mean + 1e-7))

# Variant: flatten fields too (treat each field equally by summing and dividing by F)
vrmse_flattened = torch.sqrt((mse_f.sum()) / (Var_f.sum() + 1e-7))

print('per-field VRMSE:', vrmse_per_field.tolist())
print('mean(per-field VRMSE):', round(vrmse_mean_fields.item(), 4))
print('sqrt(mean MSE / mean Var):', round(vrmse_fields_averaged_then_sqrt.item(), 4))
print('sqrt(sum MSE / sum Var):', round(vrmse_flattened.item(), 4))

print("\n=== Additional metrics ===")

print("RMSE per field:", rmse_f.tolist())
print("mean RMSE:", round(rmse_mean.item(), 4))

print("R2 per field:", r2_f.tolist())
print("mean R2:", round(r2_mean.item(), 4))

print("Relative L2 per field:", rel_l2_f.tolist())
print("mean Relative L2:", round(rel_l2_mean.item(), 4))

print("\n=== Summary ===")
print("VRMSE (mean):", round(vrmse_mean_fields.item(), 4))
print("Relative L2 (mean):", round(rel_l2_mean.item(), 4))
print("R2 (mean):", round(r2_mean.item(), 4))

Using device: mps


stream accum:   0%|          | 0/110 [00:00<?, ?it/s]/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/neuralop/layers/spectral_convolution.py:434: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [8, 128, 128, 193]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  x = torch.fft.rfftn(x, norm=self.fft_norm, dim=fft_dims)
/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/tltorch/factorized_tensors/factorized_tensors.py:66: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this wi

per-field VRMSE: [0.14225417375564575, 0.3624419569969177, 0.29539617896080017, 0.3262180685997009]
mean(per-field VRMSE): 0.2816
sqrt(mean MSE / mean Var): 0.2959
sqrt(sum MSE / sum Var): 0.2959

=== Additional metrics ===
RMSE per field: [0.14152926206588745, 0.37338346242904663, 0.2967306971549988, 0.33865511417388916]
mean RMSE: 0.2876
R2 per field: [0.9797637462615967, 0.8686358332633972, 0.9127410650253296, 0.893581748008728]
mean R2: 0.9137
Relative L2 per field: [0.1422540247440338, 0.36135050654411316, 0.29539263248443604, 0.32585152983665466]
mean Relative L2: 0.2812

=== Summary ===
VRMSE (mean): 0.2816
Relative L2 (mean): 0.2812
R2 (mean): 0.9137


In [ ]:
# per-field VRMSE: [0.14225417375564575, 0.3624419569969177, 0.29539617896080017, 0.3262180685997009]
# mean(per-field VRMSE): 0.28157758712768555
# sqrt(mean MSE / mean Var): 0.2958519756793976
# sqrt(sum MSE / sum Var): 0.29585200548171997

# === Additional metrics ===
# RMSE per field: [0.14152926206588745, 0.37338346242904663, 0.2967306971549988, 0.33865511417388916]
# mean RMSE: 0.2875746488571167
# R2 per field: [0.9797637462615967, 0.8686358332633972, 0.9127410650253296, 0.893581748008728]
# mean R2: 0.9136805534362793
# Relative L2 per field: [0.1422540247440338, 0.36135050654411316, 0.29539263248443604, 0.32585152983665466]
# mean Relative L2: 0.28121218085289

# === Summary ===
# VRMSE (mean): 0.28157758712768555
# Relative L2 (mean): 0.28121218085289
# R2 (mean): 0.9136805534362793

## rollout prediction

In [22]:
N_STEPS_INPUT = 4
N_STEPS_ROLLOUT = 4

dset = WellDataset(
    well_base_path=f'{base_path}/datasets',
    well_dataset_name='turbulent_radiative_layer_2D',
    well_split_name='test',
    n_steps_input=N_STEPS_INPUT,
    n_steps_output=N_STEPS_ROLLOUT,
    use_normalization=True,
    normalization_type=ZScoreNormalization,
)

loader = DataLoader(dset, batch_size=8, shuffle=False)

F = dset.metadata.n_fields
norm = dset.norm

mean_full = norm.flattened_means['variable'].to(device)         # (F,)
std_full  = norm.flattened_stds['variable'].to(device)          # (F,)
mean_d    = norm.flattened_means_delta['variable'].to(device)   # (F,)
std_d     = norm.flattened_stds_delta['variable'].to(device)    # (F,)

# -------------------------
# global accumulators over all rollout steps
# -------------------------
sum_err2 = torch.zeros(F, device=device)
sum_y = torch.zeros(F, device=device)
sum_y2 = torch.zeros(F, device=device)
sum_abs_err = torch.zeros(F, device=device)
rel_l2_num = torch.zeros(F, device=device)
rel_l2_den = torch.zeros(F, device=device)
count = 0

# -------------------------
# per-horizon accumulators: step 1, 2, ..., 8
# -------------------------
sum_err2_h = torch.zeros(N_STEPS_ROLLOUT, F, device=device)
sum_y_h = torch.zeros(N_STEPS_ROLLOUT, F, device=device)
sum_y2_h = torch.zeros(N_STEPS_ROLLOUT, F, device=device)
count_h = torch.zeros(N_STEPS_ROLLOUT, device=device)

with torch.no_grad():
    for batch in tqdm(loader, desc='autoregressive rollout accum'):
        x = batch['input_fields'].to(device)    # (B, 4, Lx, Ly, F)
        y = batch['output_fields'].to(device)   # (B, 8, Lx, Ly, F)

        # rolling window of normalized full frames
        current_window = x.clone()              # (B, 4, Lx, Ly, F)

        preds = []

        for t in range(N_STEPS_ROLLOUT):
            # model input: normalized full frames
            x_in = rearrange(current_window, 'B Ti Lx Ly F -> B (Ti F) Lx Ly')
            pred = model(x_in)  # predicts normalized delta
            pred = rearrange(pred, 'B (To F) Lx Ly -> B To Lx Ly F', To=1, F=F)

            last_input_full_norm = current_window[:, -1]   # (B, Lx, Ly, F)
            pred_delta_norm = pred[:, 0]                   # (B, Lx, Ly, F)

            # delta norm -> delta raw -> next full raw -> next full norm
            last_raw = last_input_full_norm * std_full + mean_full
            delta_raw = pred_delta_norm * std_d + mean_d
            next_full_raw = last_raw + delta_raw
            next_full_norm = (next_full_raw - mean_full) / std_full  # (B, Lx, Ly, F)

            preds.append(next_full_norm)

            # slide the window: drop oldest, append prediction
            current_window = torch.cat(
                [current_window[:, 1:], next_full_norm.unsqueeze(1)],
                dim=1
            )

        pred_full_norm = torch.stack(preds, dim=1)  # (B, 8, Lx, Ly, F)

        err = pred_full_norm - y
        err2 = err ** 2

        # -------------------------
        # global accumulators
        # -------------------------
        sum_err2 += err2.sum(dim=(0, 1, 2, 3))
        sum_y += y.sum(dim=(0, 1, 2, 3))
        sum_y2 += (y * y).sum(dim=(0, 1, 2, 3))

        sum_abs_err += err.abs().sum(dim=(0, 1, 2, 3))
        rel_l2_num += err2.sum(dim=(0, 1, 2, 3))
        rel_l2_den += (y ** 2).sum(dim=(0, 1, 2, 3))

        count += y.shape[0] * y.shape[1] * y.shape[2] * y.shape[3]

        # -------------------------
        # per-horizon accumulators
        # -------------------------
        for t in range(N_STEPS_ROLLOUT):
            err_t = err[:, t:t+1]   # (B,1,Lx,Ly,F)
            y_t = y[:, t:t+1]

            sum_err2_h[t] += (err_t ** 2).sum(dim=(0, 1, 2, 3))
            sum_y_h[t] += y_t.sum(dim=(0, 1, 2, 3))
            sum_y2_h[t] += (y_t ** 2).sum(dim=(0, 1, 2, 3))
            count_h[t] += y_t.shape[0] * y_t.shape[1] * y_t.shape[2] * y_t.shape[3]

# -------------------------
# global metrics over all 8 rollout steps
# -------------------------
mse_f = sum_err2 / count
Ey_f = sum_y / count
Ey2_f = sum_y2 / count
Var_f = Ey2_f - Ey_f * Ey_f

vrmse_per_field = torch.sqrt(mse_f / (Var_f + 1e-7))
vrmse_mean_fields = vrmse_per_field.mean()

rmse_f = torch.sqrt(mse_f)
mae_f = sum_abs_err / count
r2_f = 1.0 - (mse_f / (Var_f + 1e-7))
rel_l2_f = torch.sqrt(rel_l2_num / (rel_l2_den + 1e-7))

rmse_mean = rmse_f.mean()
r2_mean = r2_f.mean()
rel_l2_mean = rel_l2_f.mean()

mse_mean = mse_f.mean()
Var_mean = Var_f.mean()
vrmse_fields_averaged_then_sqrt = torch.sqrt(mse_mean / (Var_mean + 1e-7))
vrmse_flattened = torch.sqrt(mse_f.sum() / (Var_f.sum() + 1e-7))

# -------------------------
# per-horizon metrics
# -------------------------
mse_h = sum_err2_h / count_h[:, None]
Ey_h = sum_y_h / count_h[:, None]
Ey2_h = sum_y2_h / count_h[:, None]
Var_h = Ey2_h - Ey_h * Ey_h

vrmse_per_field_h = torch.sqrt(mse_h / (Var_h + 1e-7))   # (8, F)
vrmse_mean_h = vrmse_per_field_h.mean(dim=1)             # (8,)

# -------------------------
# printing
# -------------------------
print('=== AUTOREGRESSIVE ROLLOUT (4 STEPS) ===')
print('per-field VRMSE:', vrmse_per_field.tolist())
print('mean(per-field VRMSE):', vrmse_mean_fields.item())
print('sqrt(mean MSE / mean Var):', vrmse_fields_averaged_then_sqrt.item())
print('sqrt(sum MSE / sum Var):', vrmse_flattened.item())

print("\n=== Additional metrics ===")
print("RMSE per field:", rmse_f.tolist())
print("mean RMSE:", rmse_mean.item())

print("MAE per field:", mae_f.tolist())
print("mean MAE:", mae_f.mean().item())

print("R2 per field:", r2_f.tolist())
print("mean R2:", r2_mean.item())

print("Relative L2 per field:", rel_l2_f.tolist())
print("mean Relative L2:", rel_l2_mean.item())

print("\n=== Per-horizon VRMSE ===")
for t in range(N_STEPS_ROLLOUT):
    print(f"step {t+1}: mean VRMSE = {vrmse_mean_h[t].item():.6f}, per-field = {vrmse_per_field_h[t].tolist()}")

print("\n=== Summary ===")
print("VRMSE (mean):", vrmse_mean_fields.item())
print("Relative L2 (mean):", rel_l2_mean.item())
print("R2 (mean):", r2_mean.item())

autoregressive rollout accum:  99%|█████████▉| 105/106 [00:37<00:00,  2.79it/s]/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/neuralop/layers/spectral_convolution.py:434: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [6, 128, 128, 193]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  x = torch.fft.rfftn(x, norm=self.fft_norm, dim=fft_dims)
/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/neuralop/layers/spectral_convolution.py:469: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [6, 128, 128, 384]. This behav

=== AUTOREGRESSIVE ROLLOUT (4 STEPS) ===
per-field VRMSE: [0.18178503215312958, 0.43907031416893005, 0.3477603793144226, 0.39581894874572754]
mean(per-field VRMSE): 0.34110867977142334
sqrt(mean MSE / mean Var): 0.3570953607559204
sqrt(sum MSE / sum Var): 0.3570953905582428

=== Additional metrics ===
RMSE per field: [0.18077482283115387, 0.45027491450309753, 0.349534809589386, 0.4109824001789093]
mean RMSE: 0.34789174795150757
MAE per field: [0.0571470633149147, 0.18181350827217102, 0.1204911544919014, 0.1634269505739212]
mean MAE: 0.13071967661380768
R2 per field: [0.966954231262207, 0.8072172403335571, 0.8790627121925354, 0.8433273434638977]
mean R2: 0.8741403818130493
Relative L2 per field: [0.18178488314151764, 0.43752944469451904, 0.34775257110595703, 0.39528340101242065]
mean Relative L2: 0.3405875563621521

=== Per-horizon VRMSE ===
step 1: mean VRMSE = 0.278697, per-field = [0.14062070846557617, 0.3611098527908325, 0.2910498380661011, 0.3220057487487793]
step 2: mean VRMSE = 0

In [ ]:
# from autoregressive_pretrained_fno import predict_autoregressive_fno, save_rollout_plots

# out = predict_autoregressive_fno(
#     base_path="./data",
#     model_pt_path="/Users/emilfahretdinov/msc_hse/models_trained/fno_delta_lr1e-3_20ep_n_modes32/best_by_valid_rollout_vrmse_delta.pt",
#     hdf5_path="./data/datasets/turbulent_radiative_layer_2D/data/test/turbulent_radiative_layer_tcool_0.32.hdf5",
#     prediction_mode="delta",   # use "full" for full-frame training checkpoints
#     traj_id=0,
#     t_start=50,
#     t_roll=40,
#     device="mps",
#     n_modes=(32, 32)
# )
# save_rollout_plots(out, field="density", out_dir="./plots_delta")

save plots: 100%|██████████| 40/40 [00:03<00:00, 11.45it/s]


'./plots_delta/traj000_density_delta_best_by_valid_rollout_vrmse_delta'

# My full frame models

## 1 step prediction

In [2]:
import pandas as pd

In [8]:
import torch
from torch.utils.data import DataLoader
from einops import rearrange
from tqdm import tqdm
from neuralop.models import FNO as NeuralFNO
from the_well.data import WellDataset
from the_well.data.normalization import ZScoreNormalization


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def load_my_model(checkpoint_path, n_fields, device):
    ckpt = torch.load(checkpoint_path, map_location="cpu")

    if isinstance(ckpt, torch.nn.Module):
        model = ckpt
    else:
        state_dict = None
        if isinstance(ckpt, dict):
            for key in ("model_state_dict", "state_dict", "model", "net"):
                if key in ckpt and isinstance(ckpt[key], dict):
                    state_dict = ckpt[key]
                    break
            if state_dict is None and all(isinstance(v, torch.Tensor) for v in ckpt.values()):
                state_dict = ckpt

        if state_dict is None:
            raise ValueError(
                "Unsupported checkpoint format. Expected nn.Module or state_dict-like dict."
            )

        if any(k.startswith("module.") for k in state_dict.keys()):
            state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

        model = NeuralFNO(
            n_modes=(64, 64),
            in_channels=4 * n_fields,
            out_channels=1 * n_fields,
            hidden_channels=128,
            n_layers=4,
        )
        model.load_state_dict(state_dict, strict=True)

    return model.to(device).eval()


base_path = "./data"
checkpoint_path = "/Users/emilfahretdinov/msc_hse/models_trained/fno_full_lr1e-3_20ep_nmodes64_nlayers4/final_model_full_frame.pt"
device = pick_device()
print(f"Using device: {device}")

dset = WellDataset(
    well_base_path=f"{base_path}/datasets",
    well_dataset_name="turbulent_radiative_layer_2D",
    well_split_name="test",
    n_steps_input=4,
    n_steps_output=1,
    use_normalization=True,
    normalization_type=ZScoreNormalization,
)
loader = DataLoader(dset, batch_size=8, shuffle=False)
F = dset.metadata.n_fields
model = load_my_model(checkpoint_path, n_fields=F, device=device)

sum_err2 = torch.zeros(F, device=device)
sum_y = torch.zeros(F, device=device)
sum_y2 = torch.zeros(F, device=device)
count = 0

#additional metrics
sum_abs_err = torch.zeros(F, device=device)

rel_l2_num = torch.zeros(F, device=device)
rel_l2_den = torch.zeros(F, device=device)


with torch.no_grad():
    for batch in tqdm(loader, desc="test 1-step"):
        x = batch["input_fields"].to(device)
        y = batch["output_fields"].to(device)
        # model expects normalized inputs already because dset uses normalization
        x_in = rearrange(x, "B Ti Lx Ly F -> B (Ti F) Lx Ly")
        pred = model(x_in)
        pred_full_norm = rearrange(pred, "B (To F) Lx Ly -> B To Lx Ly F", To=1, F=F)
        err = pred_full_norm - y
        err2 = err ** 2  # (B,1,Lx,Ly,F)

        sum_err2 += err2.sum(dim=(0, 1, 2, 3))
        sum_y += y.sum(dim=(0, 1, 2, 3))
        sum_y2 += (y * y).sum(dim=(0, 1, 2, 3))

        # NEW netrics
        sum_abs_err += err.abs().sum(dim=(0, 1, 2, 3))
        rel_l2_num += err2.sum(dim=(0, 1, 2, 3))
        rel_l2_den += (y ** 2).sum(dim=(0, 1, 2, 3))

        count += y.shape[0] * y.shape[1] * y.shape[2] * y.shape[3]

mse_f = sum_err2 / count
Ey_f = sum_y / count
Ey2_f = sum_y2 / count
Var_f = Ey2_f - Ey_f * Ey_f

#additional metrics
# RMSE
rmse_f = torch.sqrt(mse_f)
# MAE
mae_f = sum_abs_err / count
# R^2
r2_f = 1.0 - (mse_f / (Var_f + 1e-7))
# Relative L2 error
rel_l2_f = torch.sqrt(rel_l2_num / (rel_l2_den + 1e-7))

rmse_mean = rmse_f.mean()
r2_mean = r2_f.mean()
rel_l2_mean = rel_l2_f.mean()

vrmse_per_field = torch.sqrt(mse_f / (Var_f + 1e-7))
vrmse_mean_fields = vrmse_per_field.mean()

# Variant: compute MSE and Var after averaging over fields first
mse_mean = mse_f.mean()
Var_mean = Var_f.mean()
vrmse_fields_averaged_then_sqrt = torch.sqrt(mse_mean / (Var_mean + 1e-7))

# Variant: flatten fields too (treat each field equally by summing and dividing by F)
vrmse_flattened = torch.sqrt(mse_f.sum() / (Var_f.sum() + 1e-7))

print('per-field VRMSE:', vrmse_per_field.tolist())
print('mean(per-field VRMSE):', round(vrmse_mean_fields.item(), 4))
print('sqrt(mean MSE / mean Var):', round(vrmse_fields_averaged_then_sqrt.item(), 4))
print('sqrt(sum MSE / sum Var):', round(vrmse_flattened.item(), 4))

print("\n=== Additional metrics ===")

print("RMSE per field:", rmse_f.tolist())
print("mean RMSE:", round(rmse_mean.item(), 4))

print("R2 per field:", r2_f.tolist())
print("mean R2:", round(r2_mean.item(), 4))

print("Relative L2 per field:", rel_l2_f.tolist())
print("mean Relative L2:", round(rel_l2_mean.item(), 4))

print("\n=== Summary ===")
print("VRMSE (mean):", round(vrmse_mean_fields.item(), 4))
print("Relative L2 (mean):", round(rel_l2_mean.item(), 4))
print("R2 (mean):", round(r2_mean.item(), 4))

Using device: mps


test 1-step:   0%|          | 0/110 [00:00<?, ?it/s]/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/neuralop/layers/spectral_convolution.py:434: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [8, 128, 128, 193]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  x = torch.fft.rfftn(x, norm=self.fft_norm, dim=fft_dims)
/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/tltorch/factorized_tensors/factorized_tensors.py:66: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this wil

per-field VRMSE: [0.14594188332557678, 0.39082658290863037, 0.3084844946861267, 0.34383538365364075]
mean(per-field VRMSE): 0.2973
sqrt(mean MSE / mean Var): 0.3135
sqrt(sum MSE / sum Var): 0.3135

=== Additional metrics ===
RMSE per field: [0.14519819617271423, 0.4026249647140503, 0.3098781406879425, 0.35694408416748047]
mean RMSE: 0.3037
R2 per field: [0.9787009954452515, 0.8472545742988586, 0.9048373103141785, 0.8817772269248962]
mean R2: 0.9031
Relative L2 per field: [0.14594174921512604, 0.3896496295928955, 0.30848076939582825, 0.34344902634620667]
mean Relative L2: 0.2969

=== Summary ===
VRMSE (mean): 0.2973
Relative L2 (mean): 0.2969
R2 (mean): 0.9031


In [9]:
model_name = "Full_64_4"

summary_csv_path = "results/Model Comparison (1-Step Prediction).csv"
per_field_csv_path = "results/Per-field Comparison (1-Step).csv"

summary_row = {
    "Model": model_name,
    "VRMSE ↓": round(vrmse_mean_fields.item(), 4),
    "Rel L2 ↓": round(rel_l2_mean.item(), 4),
    "R² ↑": round(r2_mean.item(), 4),
    "RMSE ↓": round(rmse_mean.item(), 4),
}

per_field_row = {
    "Model": model_name,
    "Field 1": round(vrmse_per_field[0].item(), 4),
    "Field 2": round(vrmse_per_field[1].item(), 4),
    "Field 3": round(vrmse_per_field[2].item(), 4),
    "Field 4": round(vrmse_per_field[3].item(), 4),
}

df_summary = pd.read_csv(summary_csv_path)
df_summary = df_summary[df_summary["Model"] != model_name]
df_summary = pd.concat([df_summary, pd.DataFrame([summary_row])], ignore_index=True)
df_summary.to_csv(summary_csv_path, index=False)

df_per_field = pd.read_csv(per_field_csv_path)
df_per_field = df_per_field[df_per_field["Model"] != model_name]
df_per_field = pd.concat([df_per_field, pd.DataFrame([per_field_row])], ignore_index=True)
df_per_field.to_csv(per_field_csv_path, index=False)

print(f"\nSaved metrics for model: {model_name}")


Saved metrics for model: Full_64_4


## rollout prediction

In [10]:
N_STEPS_INPUT = 4
N_STEPS_ROLLOUT = 4

dset = WellDataset(
    well_base_path=f'{base_path}/datasets',
    well_dataset_name='turbulent_radiative_layer_2D',
    well_split_name='test',
    n_steps_input=N_STEPS_INPUT,
    n_steps_output=N_STEPS_ROLLOUT,
    use_normalization=True,
    normalization_type=ZScoreNormalization,
)

loader = DataLoader(dset, batch_size=8, shuffle=False)

F = dset.metadata.n_fields
norm = dset.norm

mean_full = norm.flattened_means['variable'].to(device)         # (F,)
std_full  = norm.flattened_stds['variable'].to(device)          # (F,)
mean_d    = norm.flattened_means_delta['variable'].to(device)   # (F,)
std_d     = norm.flattened_stds_delta['variable'].to(device)    # (F,)

# -------------------------
# global accumulators over all rollout steps
# -------------------------
sum_err2 = torch.zeros(F, device=device)
sum_y = torch.zeros(F, device=device)
sum_y2 = torch.zeros(F, device=device)
sum_abs_err = torch.zeros(F, device=device)
rel_l2_num = torch.zeros(F, device=device)
rel_l2_den = torch.zeros(F, device=device)
count = 0

# -------------------------
# per-horizon accumulators: step 1, 2, ..., 4
# -------------------------
sum_err2_h = torch.zeros(N_STEPS_ROLLOUT, F, device=device)
sum_y_h = torch.zeros(N_STEPS_ROLLOUT, F, device=device)
sum_y2_h = torch.zeros(N_STEPS_ROLLOUT, F, device=device)
count_h = torch.zeros(N_STEPS_ROLLOUT, device=device)

with torch.no_grad():
    for batch in tqdm(loader, desc='autoregressive rollout accum'):
        x = batch['input_fields'].to(device)    # (B, 4, Lx, Ly, F)
        y = batch['output_fields'].to(device)   # (B, 4, Lx, Ly, F)

        # rolling window of normalized full frames
        current_window = x.clone()              # (B, 4, Lx, Ly, F)

        preds = []

        for t in range(N_STEPS_ROLLOUT):
            # model input: normalized full frames
            x_in = rearrange(current_window, 'B Ti Lx Ly F -> B (Ti F) Lx Ly')
            pred = model(x_in)  # predicts normalized delta
            next_full_norm = rearrange(pred, "B (To F) Lx Ly -> B To Lx Ly F", To=1, F=F)[:, 0]  # (B, Lx, Ly, F)

            preds.append(next_full_norm)

            # slide the window: drop oldest, append prediction
            current_window = torch.cat(
                [current_window[:, 1:], next_full_norm.unsqueeze(1)],
                dim=1
            )

        pred_full_norm = torch.stack(preds, dim=1)  # (B, 8, Lx, Ly, F)

        err = pred_full_norm - y
        err2 = err ** 2

        # -------------------------
        # global accumulators
        # -------------------------
        sum_err2 += err2.sum(dim=(0, 1, 2, 3))
        sum_y += y.sum(dim=(0, 1, 2, 3))
        sum_y2 += (y * y).sum(dim=(0, 1, 2, 3))

        sum_abs_err += err.abs().sum(dim=(0, 1, 2, 3))
        rel_l2_num += err2.sum(dim=(0, 1, 2, 3))
        rel_l2_den += (y ** 2).sum(dim=(0, 1, 2, 3))

        count += y.shape[0] * y.shape[1] * y.shape[2] * y.shape[3]

        # -------------------------
        # per-horizon accumulators
        # -------------------------
        for t in range(N_STEPS_ROLLOUT):
            err_t = err[:, t:t+1]   # (B,1,Lx,Ly,F)
            y_t = y[:, t:t+1]

            sum_err2_h[t] += (err_t ** 2).sum(dim=(0, 1, 2, 3))
            sum_y_h[t] += y_t.sum(dim=(0, 1, 2, 3))
            sum_y2_h[t] += (y_t ** 2).sum(dim=(0, 1, 2, 3))
            count_h[t] += y_t.shape[0] * y_t.shape[1] * y_t.shape[2] * y_t.shape[3]

# -------------------------
# global metrics over all 8 rollout steps
# -------------------------
mse_f = sum_err2 / count
Ey_f = sum_y / count
Ey2_f = sum_y2 / count
Var_f = Ey2_f - Ey_f * Ey_f

vrmse_per_field = torch.sqrt(mse_f / (Var_f + 1e-7))
vrmse_mean_fields = vrmse_per_field.mean()

rmse_f = torch.sqrt(mse_f)
mae_f = sum_abs_err / count
r2_f = 1.0 - (mse_f / (Var_f + 1e-7))
rel_l2_f = torch.sqrt(rel_l2_num / (rel_l2_den + 1e-7))

rmse_mean = rmse_f.mean()
r2_mean = r2_f.mean()
rel_l2_mean = rel_l2_f.mean()

mse_mean = mse_f.mean()
Var_mean = Var_f.mean()
vrmse_fields_averaged_then_sqrt = torch.sqrt(mse_mean / (Var_mean + 1e-7))
vrmse_flattened = torch.sqrt(mse_f.sum() / (Var_f.sum() + 1e-7))

# -------------------------
# per-horizon metrics
# -------------------------
mse_h = sum_err2_h / count_h[:, None]
Ey_h = sum_y_h / count_h[:, None]
Ey2_h = sum_y2_h / count_h[:, None]
Var_h = Ey2_h - Ey_h * Ey_h

vrmse_per_field_h = torch.sqrt(mse_h / (Var_h + 1e-7))   # (8, F)
vrmse_mean_h = vrmse_per_field_h.mean(dim=1)             # (8,)

# -------------------------
# printing
# -------------------------
print('=== AUTOREGRESSIVE ROLLOUT (4 STEPS) ===')
print('per-field VRMSE:', vrmse_per_field.tolist())
print('mean(per-field VRMSE):', vrmse_mean_fields.item())
print('sqrt(mean MSE / mean Var):', vrmse_fields_averaged_then_sqrt.item())
print('sqrt(sum MSE / sum Var):', vrmse_flattened.item())

print("\n=== Additional metrics ===")
print("RMSE per field:", rmse_f.tolist())
print("mean RMSE:", rmse_mean.item())

print("MAE per field:", mae_f.tolist())
print("mean MAE:", mae_f.mean().item())

print("R2 per field:", r2_f.tolist())
print("mean R2:", r2_mean.item())

print("Relative L2 per field:", rel_l2_f.tolist())
print("mean Relative L2:", rel_l2_mean.item())

print("\n=== Per-horizon VRMSE ===")
for t in range(N_STEPS_ROLLOUT):
    print(f"step {t+1}: mean VRMSE = {vrmse_mean_h[t].item():.6f}, per-field = {vrmse_per_field_h[t].tolist()}")

print("\n=== Summary ===")
print("VRMSE (mean):", vrmse_mean_fields.item())
print("Relative L2 (mean):", rel_l2_mean.item())
print("R2 (mean):", r2_mean.item())

autoregressive rollout accum:   0%|          | 0/106 [00:00<?, ?it/s]

/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/neuralop/layers/spectral_convolution.py:434: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [8, 128, 128, 193]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  x = torch.fft.rfftn(x, norm=self.fft_norm, dim=fft_dims)
/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/tltorch/factorized_tensors/factorized_tensors.py:66: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq

=== AUTOREGRESSIVE ROLLOUT (4 STEPS) ===
per-field VRMSE: [0.19030506908893585, 0.45878055691719055, 0.3522225320339203, 0.40758681297302246]
mean(per-field VRMSE): 0.3522237539291382
sqrt(mean MSE / mean Var): 0.36880457401275635
sqrt(sum MSE / sum Var): 0.36880460381507874

=== Additional metrics ===
RMSE per field: [0.1892475187778473, 0.47048816084861755, 0.3540197014808655, 0.4232011139392853]
mean RMSE: 0.3592391312122345
MAE per field: [0.05669509619474411, 0.19141849875450134, 0.12016754597425461, 0.168193519115448]
mean MAE: 0.13411866128444672
R2 per field: [0.9637839794158936, 0.7895203828811646, 0.8759393095970154, 0.8338729739189148]
mean R2: 0.8657791614532471
Relative L2 per field: [0.1903049200773239, 0.4571705162525177, 0.35221460461616516, 0.40703538060188293]
mean Relative L2: 0.35168135166168213

=== Per-horizon VRMSE ===
step 1: mean VRMSE = 0.294168, per-field = [0.14390172064304352, 0.3889789283275604, 0.3043356239795685, 0.33945387601852417]
step 2: mean VRMSE =

In [11]:
summary_csv = "results/Model Comparison (Rollout 4-step).csv"
per_field_csv = "results/Per-field Comparison (Rollout).csv"
horizon_csv = "results/Error Growth (Rollout Stability).csv"

# =========================
# 1. SUMMARY TABLE
# =========================
summary_row = {
    "Model": model_name,
    "VRMSE ↓": round(vrmse_mean_fields.item(), 4),
    "Rel L2 ↓": round(rel_l2_mean.item(), 4),
    "R² ↑": round(r2_mean.item(), 4),
    "RMSE ↓": round(rmse_mean.item(), 4),
    "MAE ↓": round(mae_f.mean().item(), 4),
}


df = pd.read_csv(summary_csv)
df = df[df["Model"] != model_name]
df = pd.concat([df, pd.DataFrame([summary_row])], ignore_index=True)
df.to_csv(summary_csv, index=False)


per_field_row = {
    "Model": model_name,
    "Field 1": round(vrmse_per_field[0].item(), 4),
    "Field 2": round(vrmse_per_field[1].item(), 4),
    "Field 3": round(vrmse_per_field[2].item(), 4),
    "Field 4": round(vrmse_per_field[3].item(), 4),
}

df_pf = pd.read_csv(per_field_csv)
df_pf = df_pf[df_pf["Model"] != model_name]
df_pf = pd.concat([df_pf, pd.DataFrame([per_field_row])], ignore_index=True)
df_pf.to_csv(per_field_csv, index=False)


# =========================
# 3. ERROR GROWTH (PER STEP)
# =========================
horizon_row = {"Model": model_name}

for t in range(N_STEPS_ROLLOUT):
    horizon_row[f"{t+1}"] = round(vrmse_mean_h[t].item(), 4)

df_h = pd.read_csv(horizon_csv)
df_h = df_h[df_h["Model"] != model_name]
df_h = pd.concat([df_h, pd.DataFrame([horizon_row])], ignore_index=True)
df_h.to_csv(horizon_csv, index=False)
# =========================
# DONE
# =========================
print(f"\nSaved rollout metrics for: {model_name}")


Saved rollout metrics for: Full_64_4


In [4]:
models_paths = [("/Users/emilfahretdinov/msc_hse/models_trained/fno_delta_lr1e-3_50ep_nmodes16_nlayers4/best_by_valid_rollout_vrmse_delta.pt", "delta_16_4"),
                ("/Users/emilfahretdinov/msc_hse/models_trained/fno_delta_lr1e-3_20ep_nmodes32_nlayers4/last_delta_model.pt", "delta_32_4"),
                ("/Users/emilfahretdinov/msc_hse/models_trained/fno_full_frame_lr5e-3_40ep_nmodes16_nlayers4/best_by_valid_rollout_vrmse.pt", "full_16_4"),
                ("/Users/emilfahretdinov/msc_hse/models_trained/fno_full_lr1e-3_20ep_nmodes32_nlayers4/final_model_full_frame.pt", "full_32_4"),
                ("/Users/emilfahretdinov/msc_hse/models_trained/fno_full_lr1e-3_20ep_nmodes32_nlayers5/final_model_full_frame.pt", "full_32_5"),
                ("/Users/emilfahretdinov/msc_hse/models_trained/fno_full_lr1e-3_20ep_nmodes64_nlayers4/final_model_full_frame.pt", "full_64_4")
                ]

In [5]:
from autoregressive_pretrained_fno import predict_autoregressive_fno, save_rollout_plots

for model_path, model_name in models_paths:
    params = model_name.split("_")
    mode = params[0]
    n_modes = int(params[1])
    n_layers = int(params[2])
    out = predict_autoregressive_fno(
        base_path="./data",
        model_pt_path=model_path,
        hdf5_path="./data/datasets/turbulent_radiative_layer_2D/data/test/turbulent_radiative_layer_tcool_0.32.hdf5",
        prediction_mode=mode,   # use "full" for full-frame training checkpoints
        n_modes=(n_modes, n_modes),
        traj_id=0,
        t_start=0,
        t_roll=97,
        device="mps",
        n_layers=n_layers,
        )
    save_rollout_plots(out, field="density", out_dir=f"./plots_{mode}/{mode}_{n_modes}_{n_layers}")

save plots: 100%|██████████| 97/97 [00:08<00:00, 11.61it/s]


In [ ]:
from autoregressive_pretrained_fno import predict_autoregressive_fno, save_rollout_plots

out = predict_autoregressive_fno(
    base_path="./data",
    model_pt_path="/Users/emilfahretdinov/msc_hse/models_trained/fno_delta_lr1e-3_50ep_nmodes16_nlayers4/best_by_valid_rollout_vrmse_delta.pt",
    hdf5_path="./data/datasets/turbulent_radiative_layer_2D/data/test/turbulent_radiative_layer_tcool_0.32.hdf5",
    prediction_mode="full",   # use "full" for full-frame training checkpoints
    n_modes=(32, 32),
    traj_id=0,
    t_start=0,
    t_roll=10,
    device="mps",
)
save_rollout_plots(out, field="density", out_dir="./plots_delta/full_32_4")

save plots: 100%|██████████| 10/10 [00:00<00:00, 12.80it/s]


'./plots_full/full_32_4/traj000_density_full_final_model_full_frame'

In [6]:
out = predict_autoregressive_fno(
    base_path="./data",
    hdf5_path = "./data/datasets/turbulent_radiative_layer_2D/data/test/turbulent_radiative_layer_tcool_0.32.hdf5",
    repo_id="polymathic-ai/FNO-turbulent_radiative_layer_2D",
    prediction_mode="delta",
    t_roll=10,
)

save_rollout_plots(out, field="density", out_dir="./plots_delta/authors_model")

/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/neuralop/layers/spectral_convolution.py:434: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [1, 128, 128, 193]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  x = torch.fft.rfftn(x, norm=self.fft_norm, dim=fft_dims)
/Users/emilfahretdinov/msc_hse/.venv/lib/python3.14/site-packages/tltorch/factorized_tensors/factorized_tensors.py:66: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq

'./plots_delta/authors_model/traj000_density_pretrained_delta'

# Make gifs

In [7]:
from pathlib import Path
from plot_functions.plot_folder_gifs import make_gifs_from_plot_roots, pngs_to_gif, list_step_pngs
repo = Path("/Users/emilfahretdinov/msc_hse")
make_gifs_from_plot_roots(
    [repo / "plots_delta", repo / "plots_full"],
    repo / "plot_gifs_out",
    max_edge=400,
    palette_colors=48,
    fps=8,
    frame_stride=1,
)

[PosixPath('/Users/emilfahretdinov/msc_hse/plot_gifs_out/plots_delta/authors_model/traj000_density_pretrained_delta/sequence.gif'),
 PosixPath('/Users/emilfahretdinov/msc_hse/plot_gifs_out/plots_delta/delta_32_4/traj000_density_delta_last_delta_model/sequence.gif'),
 PosixPath('/Users/emilfahretdinov/msc_hse/plot_gifs_out/plots_delta/delta_16_4/traj000_density_delta_best_by_valid_rollout_vrmse_delta/sequence.gif'),
 PosixPath('/Users/emilfahretdinov/msc_hse/plot_gifs_out/plots_full/full_64_4/traj000_density_full_final_model_full_frame/sequence.gif'),
 PosixPath('/Users/emilfahretdinov/msc_hse/plot_gifs_out/plots_full/full_16_4/traj000_density_full_best_by_valid_rollout_vrmse/sequence.gif'),
 PosixPath('/Users/emilfahretdinov/msc_hse/plot_gifs_out/plots_full/full_32_4/traj000_density_full_final_model_full_frame/sequence.gif'),
 PosixPath('/Users/emilfahretdinov/msc_hse/plot_gifs_out/plots_full/full_32_5/traj000_density_full_final_model_full_frame/sequence.gif')]